# Muharaf transformer: full training split, then validation

This continues the frozen ArTrOCR experiment. **One epoch over all 22,092 existing set0 training lines**, with weights updated by teacher forcing. It then predicts all 1,069 validation lines and compares those predictions with the frozen run. No test set is opened. Source images and labels are unchanged.

Choose a GPU runtime (A100 if available; L4 may take considerably longer). The model is ~0.3B parameters, so first check Colab free Drive storage for a model plus resumable optimizer checkpoint. Run in order. The first cells repeat the staging and label checks from Stage 1, so this notebook works after a Colab reset. The single-line prediction is only a setup check.

The **training cell can take a long time**. It saves a full resumable checkpoint roughly every 300 optimizer updates and prints progress every 25. The checkpoint includes optimizer state and is several GB. Keep the notebook connected if possible; Drive holds completed checkpoints. Training and validation outputs are separate from the frozen predictions.

In [ ]:
%pip -q install -U "transformers>=4.50,<6" accelerate jiwer safetensors

In [ ]:
import os, sys, torch, transformers
from google.colab import drive
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Change runtime type to GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__, 'Transformers:', transformers.__version__)

In [ ]:
from pathlib import Path
DATA_DIR = Path('/content/drive/MyDrive/Colab Notebooks/data-stuff')
assert DATA_DIR.is_dir(), f'Drive folder not found: {DATA_DIR}'
ZIP = DATA_DIR/'public_line_images.zip'
assert ZIP.is_file(), f'ZIP not found: {ZIP}'
candidate_dirs = [p.parent for p in DATA_DIR.rglob('validation.jsonl') if (p.parent/'train.jsonl').is_file()]
print('Archive:', ZIP)
print('Manifest directory candidates:', candidate_dirs)
assert len(candidate_dirs) == 1, 'Expected exactly one folder containing train.jsonl and validation.jsonl under data-stuff. Check the folder name/contents.'
MANIFESTS = candidate_dirs[0]
OUTPUT = DATA_DIR/'muharaf_transformer_runs'/'frozen_v1'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Outputs:', OUTPUT)

In [ ]:
import shutil, zipfile
LOCAL_ZIP = Path('/content/public_line_images.zip')
STAGING = Path('/content/muharaf_public')
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != ZIP.stat().st_size:
    shutil.copyfile(ZIP, LOCAL_ZIP)
with zipfile.ZipFile(LOCAL_ZIP) as z:
    names = z.namelist()
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in names), 'Archive contains unsafe names'
    assert any(n.startswith('public/') and n.endswith('.png') for n in names), 'Expected public/*.png inside ZIP'
    if not (STAGING/'public').exists():
        z.extractall(STAGING)
PUBLIC = STAGING/'public'
pngs = list(PUBLIC.rglob('*.png'))
print('Extracted PNGs:', len(pngs), 'at', PUBLIC)
assert len(pngs) == 24495, 'Expected 24,495 line images; check the ZIP and extraction.'
assert all(p.with_suffix('.txt').exists() for p in pngs), 'A line PNG is missing its TXT label.'

### Validate manifest fields and map lines to local files

The manifest may contain original paths from another computer. This cell maps by the PNG filename inside the ZIP, requires names to be unique, and checks that each manifest reference agrees with its unchanged TXT file. If your manifest uses different field names, the first printed record shows what to adapt.

In [ ]:
import json, collections
sample_record = json.loads((MANIFESTS/'train.jsonl').open(encoding='utf-8').readline())
print('Manifest field names:', list(sample_record))
print('One record (shortened):', {k:str(v)[:110] for k,v in sample_record.items()})

by_name = {}
for p in pngs:
    if p.name in by_name:
        raise ValueError(f'Duplicate image filename: {p.name}')
    by_name[p.name] = p

def pick(record, names):
    for k in names:
        if k in record and isinstance(record[k], str):
            return record[k]
    return None

def read_split(name):
    result = []
    with (MANIFESTS/f'{name}.jsonl').open(encoding='utf-8') as fh:
        for lineno, row in enumerate(fh, 1):
            r = json.loads(row)
            img = pick(r, ('image_path','image','image_file','png_path','line_image_path','line_image'))
            if not img:
                paths = [v for v in r.values() if isinstance(v,str) and v.lower().endswith('.png')]
                if len(paths) == 1: img = paths[0]
            assert img, f'No PNG field in {name}:{lineno}; inspect the printed record above'
            p = by_name.get(Path(img).name)
            assert p, f'Missing PNG for {name}:{lineno}: {img}'
            reference = p.with_suffix('.txt').read_text(encoding='utf-8')
            recorded = pick(r, ('text','transcription','ground_truth','gt','label_text','reference'))
            if recorded is not None:
                assert recorded == reference, f'Manifest text differs from original label for {p.name}'
            result.append({'image':p, 'reference':reference, 'source':r})
    assert len({x['image'] for x in result}) == len(result), f'Duplicate {name} image'
    return result

train, valid = read_split('train'), read_split('validation')
assert len(train) == 22092 and len(valid) == 1069, (len(train),len(valid))
assert not ({x['image'] for x in train} & {x['image'] for x in valid})
print('Training lines:',len(train),'Validation lines:',len(valid))
print('Example reference:',repr(valid[0]['reference'][:140]))

### Load transformer and verify it can encode every training label

The tokenizer check prevents silent dropping or changing of harakat. No checkpoint weights are updated in this stage. The author's card uses 512×102 images and positional interpolation. We reproduce that image sizing without its optional binarization or morphology.

In [ ]:
from transformers import VisionEncoderDecoderModel, TrOCRProcessor
MODEL_ID = 'BushraAlmod03/ArTrOCR-HTR'
processor = TrOCRProcessor.from_pretrained(MODEL_ID)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID).to('cuda')
assert processor.image_processor.do_resize is False, 'Review processor resize behavior before proceeding.'
assert model.config.decoder_start_token_id is not None and model.config.pad_token_id is not None
special = processor.tokenizer.all_special_ids
bad = []
max_label_tokens = 0
for i, x in enumerate(train):
    ids = processor.tokenizer(x['reference'], add_special_tokens=False)['input_ids']
    max_label_tokens = max(max_label_tokens, len(ids))
    back = processor.tokenizer.decode(ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)
    if back != x['reference'] or any(v in special for v in ids):
        bad.append((i,repr(x['reference'][:50]),repr(back[:50])))
print('Tokenizer non-roundtrip labels:',len(bad),'examples:',bad[:5])
assert not bad, 'Stop: tokenizer does not preserve every training label exactly.'
MAX_NEW_TOKENS = max_label_tokens + 8
decoder_limit = getattr(model.config.decoder, 'max_position_embeddings', None)
print('Longest training reference:',max_label_tokens,'tokens; generation allowance:',MAX_NEW_TOKENS,'decoder limit:',decoder_limit)
assert decoder_limit is None or MAX_NEW_TOKENS + 2 <= decoder_limit, 'A label exceeds the decoder position limit; review long lines before scoring.'

# The model card explicitly enables encoder position interpolation for generation.
if not hasattr(model.encoder, '_muharaf_original_forward'):
    model.encoder._muharaf_original_forward = model.encoder.forward
    def interpolated_forward(pixel_values, *args, **kwargs):
        kwargs['interpolate_pos_encoding'] = True
        return model.encoder._muharaf_original_forward(pixel_values, *args, **kwargs)
    model.encoder.forward = interpolated_forward
model.eval()
print('Model loaded; weights frozen for inference.')

In [ ]:
from PIL import Image
import numpy as np
def prepare_image(path):
    with Image.open(path) as src:
        img = src.convert('RGB')
    w,h = img.size
    target_w,target_h = 512,102
    new_w = max(1,round(w * target_h / h))
    img = img.resize((new_w,target_h),Image.Resampling.LANCZOS)
    if new_w > target_w:
        img = img.resize((target_w,target_h),Image.Resampling.LANCZOS)
        new_w = target_w
    canvas = Image.new('RGB',(target_w,target_h),(255,255,255))
    canvas.paste(img,(0,0))
    return canvas

def predict(item):
    im = prepare_image(item['image'])
    pixels = processor(images=im,return_tensors='pt').pixel_values.to('cuda')
    with torch.inference_mode():
        # One decoder-start token plus the same generated-token allowance.
        # Passing max_length alone avoids the checkpoint's max_length=20 warning.
        ids = model.generate(pixels,max_length=MAX_NEW_TOKENS+1,num_beams=1)
    return processor.batch_decode(ids,skip_special_tokens=True,clean_up_tokenization_spaces=False)[0]

print('Example:',valid[0]['image'].name)
print('Reference:',repr(valid[0]['reference']))
print('Prediction:',repr(predict(valid[0])))

### Fixed experiment choices

One epoch, all available training lines, learning rate 1e-5, per-device batch 2 and eight batches accumulated per optimizer step (effective batch 16); seed 37; no rotations, thresholding, or text cleanup. These are a single declared candidate, not a hidden search. Dynamic label padding masks only pad tokens from the loss. No sample from validation or test trains the model.

In [ ]:
import math, random, time, shutil, os
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_

TRAIN_OUTPUT = DATA_DIR/'muharaf_transformer_runs'/'finetune_v1'
TRAIN_OUTPUT.mkdir(parents=True, exist_ok=True)
CHECKPOINT = TRAIN_OUTPUT/'checkpoint_latest'
FROZEN_OUTPUT = DATA_DIR/'muharaf_transformer_runs'/'frozen_v1'
FROZEN_PREDS = FROZEN_OUTPUT/'validation_predictions.jsonl'
assert FROZEN_PREDS.is_file(), 'Frozen validation predictions missing; finish Stage 1 first.'

SEED = 37
BATCH_SIZE = 2
GRAD_ACCUM = 8
EPOCHS = 1
LEARNING_RATE = 1e-5
SAVE_EVERY = 300
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

config = {'model':MODEL_ID,'split':'public_1100/set0','train_lines':len(train),
          'seed':SEED,'batch_size':BATCH_SIZE,'grad_accum':GRAD_ACCUM,
          'epochs':EPOCHS,'lr':LEARNING_RATE,'save_every':SAVE_EVERY,
          'transformers':transformers.__version__,'torch':torch.__version__,
          'max_new_tokens':MAX_NEW_TOKENS,'image_size':[512,102]}
config_file = TRAIN_OUTPUT/'run_config.json'
if config_file.is_file():
    assert json.loads(config_file.read_text(encoding='utf-8')) == config, 'Existing training run has different settings. Use a new run folder.'
else:
    config_file.write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
print('Training:',len(train),'lines; GPU:',torch.cuda.get_device_name(0))
print('Checkpoint directory:',CHECKPOINT)
print('Approximate optimizer updates:',math.ceil(math.ceil(len(train)/BATCH_SIZE)/GRAD_ACCUM))

In [ ]:
class Lines(Dataset):
    def __init__(self, entries): self.entries = entries
    def __len__(self): return len(self.entries)
    def __getitem__(self, index):
        item = self.entries[index]
        image = prepare_image(item['image'])
        pixels = processor(images=image,return_tensors='pt').pixel_values[0]
        ids = processor.tokenizer(item['reference'], add_special_tokens=True)['input_ids']
        assert len(ids) <= MAX_NEW_TOKENS + 2
        return pixels, ids

def collate(batch):
    pixels = torch.stack([x[0] for x in batch])
    length = max(len(x[1]) for x in batch)
    labels = torch.full((len(batch),length),-100,dtype=torch.long)
    for i, (_,ids) in enumerate(batch): labels[i,:len(ids)] = torch.tensor(ids,dtype=torch.long)
    return pixels, labels

# Fixed shuffle independent of global torch RNG; recreating it gives the same order on resume.
order_generator = torch.Generator().manual_seed(SEED)
loader = DataLoader(Lines(train), batch_size=BATCH_SIZE, shuffle=True,
                    generator=order_generator, num_workers=2, pin_memory=True,
                    collate_fn=collate)
micro_total = len(loader)
update_total = math.ceil(micro_total/GRAD_ACCUM)
warmup = max(1,math.ceil(update_total*0.05))

optimizer = torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=0.01,foreach=False)
def lr_factor(step):
    if step < warmup: return (step+1)/warmup
    return max(0.0,0.5*(1+math.cos(math.pi*(step-warmup)/max(1,update_total-warmup))))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda=lr_factor)
use_bf16 = torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
scaler = torch.amp.GradScaler('cuda',enabled=not use_bf16)
print('AMP dtype:',amp_dtype,'microbatches:',micro_total,'optimizer updates:',update_total)

### Train, saving a resumable checkpoint

Checkpoints are first assembled in Colab's temporary filesystem, copied to a new Drive folder, and then rotated into place. A reset during copying may leave `checkpoint_staging` or `checkpoint_backup`; the next run chooses the highest completed step. Only checkpoints created by this notebook should be loaded. If you run out of GPU memory, stop and tell me the error before changing settings: a new run folder is required for different batch sizes.

In [ ]:
import json

def valid_checkpoint(path):
    try:
        meta = json.loads((path/'state.json').read_text(encoding='utf-8'))
        return (path/'model').is_dir() and (path/'optimizer.pt').is_file() and (path/'rng.pt').is_file() and meta['step'] <= update_total and meta['micro_done'] <= micro_total
    except (OSError,KeyError,ValueError):
        return False

candidates = [p for p in [CHECKPOINT,TRAIN_OUTPUT/'checkpoint_staging',TRAIN_OUTPUT/'checkpoint_backup'] if valid_checkpoint(p)]
resume = max(candidates,key=lambda p:json.loads((p/'state.json').read_text())['step']) if candidates else None
step = 0
micro_done = 0
if resume:
    meta = json.loads((resume/'state.json').read_text(encoding='utf-8'))
    assert meta['config'] == config, 'Checkpoint does not match this run configuration.'
    model = VisionEncoderDecoderModel.from_pretrained(resume/'model').to('cuda')
    if not hasattr(model.encoder, '_muharaf_original_forward'):
        model.encoder._muharaf_original_forward = model.encoder.forward
        def interpolated_forward(pixel_values,*args,**kwargs):
            kwargs['interpolate_pos_encoding'] = True
            return model.encoder._muharaf_original_forward(pixel_values,*args,**kwargs)
        model.encoder.forward = interpolated_forward
    optimizer = torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=0.01,foreach=False)
    optimizer.load_state_dict(torch.load(resume/'optimizer.pt',map_location='cpu',weights_only=False)['optimizer'])
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda=lr_factor)
    scheduler.load_state_dict(torch.load(resume/'optimizer.pt',map_location='cpu',weights_only=False)['scheduler'])
    scaler.load_state_dict(torch.load(resume/'optimizer.pt',map_location='cpu',weights_only=False)['scaler'])
    state = torch.load(resume/'rng.pt',map_location='cpu',weights_only=False)
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['cpu'])
    torch.cuda.set_rng_state_all(state['cuda'])
    step,micro_done = meta['step'],meta['micro_done']
    print('Resuming',resume,'at update',step,'after',micro_done,'microbatches')
else:
    print('Starting fine-tuning from frozen checkpoint.')

def save_checkpoint():
    local = Path('/content/muharaf_training_checkpoint')
    if local.exists(): shutil.rmtree(local)
    local.mkdir()
    model.save_pretrained(local/'model',safe_serialization=True)
    processor.save_pretrained(local/'model')
    torch.save({'optimizer':optimizer.state_dict(), 'scheduler':scheduler.state_dict(),
                'scaler':scaler.state_dict()},local/'optimizer.pt')
    torch.save({'python':random.getstate(),'numpy':np.random.get_state(),
                'cpu':torch.get_rng_state(),'cuda':torch.cuda.get_rng_state_all()},local/'rng.pt')
    (local/'state.json').write_text(json.dumps({'config':config,'step':step,'micro_done':micro_done},indent=2))
    stage = TRAIN_OUTPUT/'checkpoint_staging'
    backup = TRAIN_OUTPUT/'checkpoint_backup'
    if stage.exists(): shutil.rmtree(stage)
    shutil.copytree(local,stage)
    assert valid_checkpoint(stage)
    if backup.exists(): shutil.rmtree(backup)
    if CHECKPOINT.exists(): CHECKPOINT.rename(backup)
    stage.rename(CHECKPOINT)
    if backup.exists(): shutil.rmtree(backup)
    print('Saved resumable checkpoint at optimizer update',step)

model.train()
optimizer.zero_grad(set_to_none=True)
order_generator.manual_seed(SEED)
started = time.monotonic()
for index,(pixels,labels) in enumerate(loader):
    if index < micro_done:
        continue
    pixels = pixels.to('cuda',non_blocking=True)
    labels = labels.to('cuda',non_blocking=True)
    remaining = micro_total - (index//GRAD_ACCUM)*GRAD_ACCUM
    divisor = min(GRAD_ACCUM,remaining)
    with torch.autocast(device_type='cuda',dtype=amp_dtype):
        loss = model(pixel_values=pixels,labels=labels).loss / divisor
    if not torch.isfinite(loss): raise RuntimeError(f'Nonfinite loss at microbatch {index}')
    scaler.scale(loss).backward()
    micro_done = index+1
    if micro_done % GRAD_ACCUM == 0 or micro_done == micro_total:
        scaler.unscale_(optimizer)
        clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        step += 1
        if step % 25 == 0 or step == update_total:
            print(f'update {step}/{update_total}, loss×accum {float(loss)*divisor:.3f}, elapsed {time.monotonic()-started:.0f}s')
        if step % SAVE_EVERY == 0 or step == update_total:
            save_checkpoint()

assert step == update_total and micro_done == micro_total
print('Training complete. Updates:',step,'time seconds:',round(time.monotonic()-started,1))

### Evaluate the trained checkpoint against the frozen predictions

This cell reads the final saved checkpoint from Drive, produces validation predictions and a report. It keeps the original frozen baseline files and all Muharaf inputs unchanged. It intentionally avoids examining the test split.

In [ ]:
from jiwer import cer, wer
assert valid_checkpoint(CHECKPOINT)
assert json.loads((CHECKPOINT/'state.json').read_text())['step'] == update_total, 'Training not complete; resume the prior cell first.'
model.eval()
PREDICTIONS = TRAIN_OUTPUT/'validation_predictions.jsonl'
existing = {}
if PREDICTIONS.exists():
    for line in PREDICTIONS.open(encoding='utf-8'):
        r = json.loads(line)
        existing[r['image_name']] = r
with PREDICTIONS.open('a',encoding='utf-8') as out:
    for idx,item in enumerate(valid,1):
        name = item['image'].name
        if name in existing:
            assert existing[name]['reference'] == item['reference']
            continue
        row = {'image_name':name,'reference':item['reference'],'prediction':predict(item)}
        out.write(json.dumps(row,ensure_ascii=False)+'\n')
        out.flush()
        existing[name] = row
        if idx%25 == 0: print('Validation',idx,'/',len(valid))

frozen = {}
for line in FROZEN_PREDS.open(encoding='utf-8'):
    row = json.loads(line)
    assert row['image_name'] not in frozen
    frozen[row['image_name']] = row
assert len(existing) == len(frozen) == len(valid) == 1069
names = [x['image'].name for x in valid]
refs = [existing[n]['reference'] for n in names]
assert refs == [frozen[n]['reference'] for n in names]
marks = set('ًٌٍَُِّْ')
remove_marks = lambda s: ''.join(c for c in s if c not in marks)
def measure(hyps):
    return {'raw_cer':cer(refs,hyps), 'wer':wer(refs,hyps),
            'cer_without_basic_harakat':cer(list(map(remove_marks,refs)),list(map(remove_marks,hyps))),
            'reference_basic_harakat':sum(c in marks for s in refs for c in s),
            'predicted_basic_harakat':sum(c in marks for s in hyps for c in s)}
report = {'model':MODEL_ID,'split':'public_1100/set0 validation',
          'lines':len(valid),'training_updates':step,'frozen':measure([frozen[n]['prediction'] for n in names]),
          'fine_tuned':measure([existing[n]['prediction'] for n in names]),
          'note':'Harakat totals are output frequency, not counts of correct matches.'}
(TRAIN_OUTPUT/'validation_comparison.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(report,ensure_ascii=False,indent=2))
print('Saved:',TRAIN_OUTPUT)